In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Đọc file dữ liệu Online Retail (Nên đảm bảo đã cài openpyxl và mlxtend)
df_retail = pd.read_excel('Online Retail.xlsx', engine='openpyxl')

# 2. Tiền xử lý dữ liệu cơ bản
df_retail['Description'] = df_retail['Description'].str.strip()
df_retail.dropna(subset=['InvoiceNo'], inplace=True)
df_retail['InvoiceNo'] = df_retail['InvoiceNo'].astype('str')

# 3. Loại bỏ hóa đơn tín dụng (chứa ký tự 'C')
df_retail = df_retail[~df_retail['InvoiceNo'].str.contains('C', na=False)]

# 4. Lọc dữ liệu nước Đức ('Germany') và gom nhóm theo InvoiceNo và Description
basket_germany = (
    df_retail[df_retail['Country'] == 'Germany']
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum().unstack().reset_index().fillna(0)
    .set_index('InvoiceNo')
)

# 5. Chuyển đổi dữ liệu sang dạng One-Hot Encoding (0 và 1)
def encode_units(x):
    return 1 if x >= 1 else 0

basket_germany = basket_germany.map(encode_units)

# 6. Xóa cột 'POSTAGE' nếu có
if 'POSTAGE' in basket_germany.columns:
    basket_germany.drop('POSTAGE', inplace=True, axis=1)

# 7. Tìm tập phổ biến bằng thuật toán Apriori với min_sup = 5% (0.05)
frequent_itemsets_germany = apriori(basket_germany, min_support=0.05, use_colnames=True)

# 8. Tạo các luật kết hợp với min_conf = 50% (0.50)
rules_germany = association_rules(frequent_itemsets_germany, metric="confidence", min_threshold=0.50)

# 9. In kết quả các luật kết hợp tìm được
print(f"=== Số lượng luật tìm được cho nước Đức (Germany): {len(rules_germany)} ===")
for idx, row in rules_germany.iterrows():
    print(f"{list(row['antecedents'])} ==> {list(row['consequents'])} | Sup: {row['support']:.4f}, Conf: {row['confidence']:.4f}")

=== Số lượng luật tìm được cho nước Đức (Germany): 8 ===
['PLASTERS IN TIN CIRCUS PARADE'] ==> ['PLASTERS IN TIN WOODLAND ANIMALS'] | Sup: 0.0678, Conf: 0.5849
['PLASTERS IN TIN SPACEBOY'] ==> ['PLASTERS IN TIN WOODLAND ANIMALS'] | Sup: 0.0613, Conf: 0.5714
['PLASTERS IN TIN WOODLAND ANIMALS'] ==> ['ROUND SNACK BOXES SET OF4 WOODLAND'] | Sup: 0.0744, Conf: 0.5397
['RED RETROSPOT CHARLOTTE BAG'] ==> ['WOODLAND CHARLOTTE BAG'] | Sup: 0.0591, Conf: 0.8438
['ROUND SNACK BOXES SET OF 4 FRUITS'] ==> ['ROUND SNACK BOXES SET OF4 WOODLAND'] | Sup: 0.1313, Conf: 0.8333
['ROUND SNACK BOXES SET OF4 WOODLAND'] ==> ['ROUND SNACK BOXES SET OF 4 FRUITS'] | Sup: 0.1313, Conf: 0.5357
['SPACEBOY LUNCH BOX'] ==> ['ROUND SNACK BOXES SET OF4 WOODLAND'] | Sup: 0.0700, Conf: 0.6809
['WOODLAND CHARLOTTE BAG'] ==> ['ROUND SNACK BOXES SET OF4 WOODLAND'] | Sup: 0.0635, Conf: 0.5000


C:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
